In [1]:
import os


In [2]:
%pwd

'c:\\Users\\Harsha vardhan\\OneDrive\\Desktop\\TextSummarization-Project\\research'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_path: Path

In [4]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        project_root = Path(CONFIG_FILE_PATH).resolve().parent.parent

        root_dir = project_root / Path(config.root_dir)
        data_path = project_root / Path(config.data_path)
        model_path = project_root / Path(config.model_path)
        tokenizer_path = project_root / Path(config.tokenizer_path)
        metric_file_path = project_root / Path(config.metric_file_path)

        create_directories([root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=root_dir,
            data_path=data_path,
            model_path=model_path,
            tokenizer_path=tokenizer_path,
            metric_file_path=metric_file_path
        )

        return model_evaluation_config

In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import evaluate
import torch
import pandas as pd
import os
from tqdm import tqdm

c:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer, batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu", column_text="article", column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):
            inputs = tokenizer(article_batch, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8, num_beams=8, max_length=128
            )
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True) for s in summaries]
            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        score = metric.compute()
        return score

    def _resolve_path(self, path: Path) -> str:
        candidate = Path(path)
        if candidate.is_absolute():
            return str(candidate)

        project_root = Path(CONFIG_FILE_PATH).resolve().parent.parent
        return str((project_root / candidate).resolve())

    def _normalize_model_dir(self, model_dir: str) -> str:
        resolved_dir = Path(model_dir)
        target_file = resolved_dir / "model.safetensors"

        if target_file.exists():
            return str(resolved_dir)

        shard_files = sorted(resolved_dir.glob("model-*.safetensors"))
        if len(shard_files) == 1:
            shard_files[0].replace(target_file)

        return str(resolved_dir)

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self._resolve_path(self.config.tokenizer_path), local_files_only=True)
        model_dir = self._normalize_model_dir(self._resolve_path(self.config.model_path))
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_dir, local_files_only=True).to(device)

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(dataset_samsum_pt["test"][0:10], rouge_metric, model_pegasus, tokenizer, batch_size=2, column_text="dialogue", column_summary="summary")

        rouge_dict = {
            rn: (score[rn].mid.fmeasure if hasattr(score[rn], "mid") else float(score[rn]))
            for rn in rouge_names
        }

        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        df.to_csv(self.config.metric_file_path, index=False)

In [8]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.evaluate()

except Exception as e:
    raise e

[2026-03-10 14:31:07,681: INFO: common: yaml file: C:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\config\config.yaml loaded successfully]
[2026-03-10 14:31:07,686: INFO: common: yaml file: C:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\params.yaml loaded successfully]
[2026-03-10 14:31:07,689: INFO: common: created directory at: artifacts]
[2026-03-10 14:31:07,690: INFO: common: created directory at: C:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\artifacts\model_evaluation]


Loading weights: 100%|██████████| 682/682 [00:02<00:00, 313.39it/s, Materializing param=model.shared.weight]                                   
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
100%|██████████| 5/5 [04:47<00:00, 57.41s/it]

[2026-03-10 14:36:01,006: INFO: rouge_scorer: Using default tokenizer.]
